In [ ]:
import os
import pickle
import multiprocessing
import numpy as np
from joblib import Parallel, delayed
from Library.config import BASE_PARAMS
from Library import run_trajectory_grid

def generate_unified_trajectories(optimal_gamma, min_time=30):
    """Generates trajectories strictly constrained to a healthy baseline."""
    phsc_range = np.arange(0.95, 1.00, 0.01)
    coef_range = np.arange(0.25, 60, 0.1)
    y0_healthy = [2.42105 * 10**11, 4 * 10**8]
    n_cores = multiprocessing.cpu_count()
    phsc_chunks = np.array_split(phsc_range, n_cores)
    
    results = Parallel(n_jobs=-1)(
        delayed(run_trajectory_grid.run_trajectory_grid)(
            chunk, coef_range, BASE_PARAMS.copy(), optimal_gamma, y0_healthy, min_time=min_time
        ) for chunk in phsc_chunks
    )
    return [df for sublist in results for df in (sublist if isinstance(sublist, list) else [sublist])]

if __name__ == '__main__':
    os.makedirs("IntermediateResults", exist_ok=True)
    VENEX_GAMMA = 7.5e-07
    MOFFITT_GAMMA = 7.5e-07
    
    unified_venex_runs = generate_unified_trajectories(VENEX_GAMMA, min_time=30)
    unified_moffitt_runs = generate_unified_trajectories(MOFFITT_GAMMA, min_time=30)
    print("\nSaving Cohorts...")
    with open("IntermediateResults/venex_best_gamma_runs.pkl", "wb") as f:
        pickle.dump(unified_venex_runs, f, protocol=pickle.HIGHEST_PROTOCOL)
    with open("IntermediateResults/moffitt_best_gamma_runs.pkl", "wb") as f:
        pickle.dump(unified_moffitt_runs, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("Pipeline Complete! Trajectories successfully saved to IntermediateResults/")